In [1]:
import sys
from pathlib import Path

# Add project root directory (hackathon_rag) to Python path
# Assumes your notebook is located at hackathon_rag/notebooks/your_notebook.ipynb
# If your notebook is in the root (hackathon_rag/), use: Path.cwd()
project_root = Path.cwd().parent if Path.cwd().name != "hackathon_rag" else Path.cwd()
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

import sys
from pathlib import Path

# Automatically finds the 'hackathon_rag' root folder regardless of notebook location
current_dir = Path.cwd()
project_root = current_dir.parents[0] if current_dir.name != "hackathon_rag" else current_dir

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print(f"Added to sys.path: {project_root}")

Added to sys.path: c:\Users\dell\OneDrive\Desktop\c++ folder\Ai hacathon creativa


In [2]:
import src.config as config
from src.config import CHUNK_OVERLAP, CHUNK_SIZE, DOCUMENTS_DIR, EMBEDDING_MODEL, LLM_MODEL
from src.vectorstore.vector_store import VectorStore, vector_store
from src.embedder import Embedder, embedder

c:\Users\dell\OneDrive\Desktop\c++ folder\Ai hacathon creativa\.venv-1\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 3426.66it/s]


In [3]:
print(CHUNK_OVERLAP, CHUNK_SIZE, DOCUMENTS_DIR, EMBEDDING_MODEL, LLM_MODEL)

50 300 C:\Users\dell\OneDrive\Desktop\c++ folder\Ai hacathon creativa\src\assets\documents NeuML/pubmedbert-base-embeddings medllama2:latest


### Loading json and converting it into list of dictionaries 

In [4]:
import json

# Path to your JSON file
json_file_path = r"C:\Users\dell\OneDrive\Desktop\c++ folder\Ai hacathon creativa\src\assests\malaria_book_chunks_cleaned_100_30.json"

# Read and load the JSON file
with open(json_file_path, "r", encoding="utf-8") as file:
    chunks: list[dict] = json.load(file)

# Verify the data
print(f"Loaded {len(chunks)} items.")
print("First item:", chunks[0])

Loaded 466 items.
First item: {'text': 'Guidelines for the treatment of malaria – 3rd edition. 1.Malaria – drug therapy. 2.Malaria – diagnosis. 3.Antimalarials – administration and dosage. 4. Drug Therapy, Combination. 5.Guideline. I.World Health Organization. ISBN 978 92 4 154912 7 (NLM classification: WC 770)', 'page': 3, 'section': 'WHO Library Cataloguing-in-Publication Data'}


In [5]:
chunks = [item for item in chunks if item.get("page") != 3]

### Converting the chunks file into documents for the chromadb 

In [6]:
from langchain_core.documents import Document

documents = []

for item in chunks:
    content = item.get("text", "")

    metadata = {
        "page": item.get("page"),
        "section": item.get("section")
    }

    doc = Document(page_content=content, metadata=metadata)
    documents.append(doc)

print("Sample Document:")
print(f"Content: {documents[0].page_content[:100]}...")
print(f"Metadata: {documents[0].metadata}")

Sample Document:
Content: **Artemisinin-based combination therapy (ACT).** A combination of an artemisinin derivative with a l...
Metadata: {'page': 6, 'section': 'Glossary'}


In [7]:
documents

[Document(metadata={'page': 6, 'section': 'Glossary'}, page_content='**Artemisinin-based combination therapy (ACT).** A combination of an artemisinin derivative with a longer-acting antimalarial that has a different mode of action. **Asexual cycle.** The life cycle of the malaria parasite in the host, from merozoite invasion of red blood cells to schizont rupture (merozoite \uf08e ring stage \uf08e trophozoite \uf08e schizont \uf08e merozoites). The duration is approximately 24 h in _Plasmodium knowlesi_ , 48 h in _P. falciparum_ , _P. ovale_ and _P. vivax_ and 72 h in _P. malariae_ . **Asexual parasitaemia.** The presence of asexual parasites in host red blood cells.'),
 Document(metadata={'page': 6, 'section': 'Glossary'}, page_content='The level of asexual parasitaemia determined by microscopy can be expressed in several ways: the percentage of infected red blood cells, the number of infected red cells per unit volume of blood, the number of parasites seen in one field on high power

### Creating langchain_chromadb 

In [8]:
from langchain_chroma import Chroma

try:
    vector_store.delete_collection()
except Exception:
    pass


vector_store = Chroma.from_documents(
    documents=documents,
    collection_name="malaria_docs",  # Name of the collection in Chroma
    embedding=embedder,
    persist_directory="./chroma_langchain_db",  # Where to save data locally, remove if not necessary
    collection_metadata={"hnsw:space": "cosine"}
)

### testing similarity from chromadb 

In [9]:
query = "What are the recommended treatments for malaria?"

# Retrieve top 3 relevant chunks
results = vector_store.similarity_search_with_score(query, k=3)

for idx, (doc, score) in enumerate(results, start=1):
    similarity_score = 1 - score
    print(f"--- Result {idx} ---")
    print(f"Cosine Similarity Score: {similarity_score:.4f} (Distance: {score:.4f})")
    print(f"Page: {doc.metadata.get('page')} | Section: {doc.metadata.get('section')}")
    print(f"Content: {doc.page_content[:200]}...")
    print()

--- Result 1 ---
Cosine Similarity Score: 0.7116 (Distance: 0.2884)
Page: 19 | Section: 1.2 | OBJECTIVES
Content: The objectives of _the Guidelines_ are to: - assist policy-makers to design and refine effective national treatment policies on the basis of the best available evidence; - help hospital and clinical c...

--- Result 2 ---
Cosine Similarity Score: 0.7116 (Distance: 0.2884)
Page: 19 | Section: 1.2 | OBJECTIVES
Content: The objectives of _the Guidelines_ are to: - assist policy-makers to design and refine effective national treatment policies on the basis of the best available evidence; - help hospital and clinical c...

--- Result 3 ---
Cosine Similarity Score: 0.6819 (Distance: 0.3181)
Page: 19 | Section: 1.3 | SCOPE
Content: _The Guidelines_ provide a framework for designing specific, detailed national treatment protocols, taking into account local patterns of resistance to antimalarial drugs and health service capacity. ...



### Trying out cross encoder embedding models (made specifically for query vs passage matching)

In [10]:
vector_store._collection.count()

920

In [11]:
pip install requirements

Note: you may need to restart the kernel to use updated packages.


ERROR: Could not find a version that satisfies the requirement requirements (from versions: none)

[notice] A new release of pip is available: 25.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip
ERROR: No matching distribution found for requirements


In [12]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

# 1. Download and load BGE or E5 from Hugging Face
embedder = HuggingFaceEmbeddings(
    model_name="NeuML/pubmedbert-base-embeddings", # Downloads automatically on first run
    model_kwargs={"device": "cpu"}       # Use "cuda" if you have a GPU
)


try :
    vector_store.delete_collection()
except Exception:
    pass

# 2. Pass directly to Chroma
vector_store = Chroma.from_documents(
    documents=documents,
    embedding=embedder,                 # Pass the model object here
    collection_name="malaria_docs",
    persist_directory="./chroma_langchain_db",
    collection_metadata={"hnsw:space": "cosine"}
)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 3096.06it/s]


In [13]:
vector_store._collection.count()

460

In [14]:
query = "What are the recommended treatments for malaria?"

# Retrieve top 3 relevant chunks
results = vector_store.similarity_search_with_score(query, k=3)

for idx, (doc, score) in enumerate(results, start=1):
    similarity_score = 1 - score
    print(f"--- Result {idx} ---")
    print(f"Cosine Similarity Score: {similarity_score:.4f} (Distance: {score:.4f})")
    print(f"Page: {doc.metadata.get('page')} | Section: {doc.metadata.get('section')}")
    print(f"Content: {doc.page_content[:200]}...")
    print()

--- Result 1 ---
Cosine Similarity Score: 0.7116 (Distance: 0.2884)
Page: 19 | Section: 1.2 | OBJECTIVES
Content: The objectives of _the Guidelines_ are to: - assist policy-makers to design and refine effective national treatment policies on the basis of the best available evidence; - help hospital and clinical c...

--- Result 2 ---
Cosine Similarity Score: 0.6819 (Distance: 0.3181)
Page: 19 | Section: 1.3 | SCOPE
Content: _The Guidelines_ provide a framework for designing specific, detailed national treatment protocols, taking into account local patterns of resistance to antimalarial drugs and health service capacity. ...

--- Result 3 ---
Cosine Similarity Score: 0.6552 (Distance: 0.3448)
Page: 93 | Section: 8.1.2 |  MANAGEMENT OF UNCOMPLICATED FALCIPARUM MALARIA DURING EPIDEMICS
Content: The principles of treatment of uncomplicated malaria are the same as those outlined in section 4. Active case detection should be undertaken to ensure that as many patients as possible receive adeq

In [15]:
query = "risk for death from malaria"

# Retrieve top 3 relevant chunks
results = vector_store.similarity_search_with_score(query, k=6)

for idx, (doc, score) in enumerate(results, start=1):
    similarity_score = 1 - score
    print(f"--- Result {idx} ---")
    print(f"Cosine Similarity Score: {similarity_score:.4f} (Distance: {score:.4f})")
    print(f"Page: {doc.metadata.get('page')} | Section: {doc.metadata.get('section')}")
    print(f"Content: {doc.page_content[:200]}...")
    print()

--- Result 1 ---
Cosine Similarity Score: 0.6472 (Distance: 0.3528)
Page: 76 | Section: _Pre-referral treatment options_
Content: Any patient with malaria who is unable to take oral medications reliably, shows any evidence of vital organ dysfunction or has a high parasite count is at increased risk for dying. The exact risk depe...

--- Result 2 ---
Cosine Similarity Score: 0.5753 (Distance: 0.4247)
Page: 85 | Section: Other considerations
Content: The risk for death from severe malaria is greatest in the first 24 h, yet, in most malaria-endemic countries, the transit time between referral and arrival at a health facility where intravenous treat...

--- Result 3 ---
Cosine Similarity Score: 0.5727 (Distance: 0.4273)
Page: 77 | Section: 7.2 | THERAPEUTIC OBJECTIVES
Content: The main objective of the treatment of severe malaria is to prevent the patient from dying. Secondary objectives are prevention of disabilities and prevention of recrudescent infection. Death from sev...

--- Result 

### Evaluation 

In [16]:
import json

test_data = [
    {
        "question": "What are the five recommended ACTs for treating uncomplicated P. falciparum malaria?",
        "answer_section_content": "The five ACTs recommended for treatment of uncomplicated P. falciparum malaria are: - artemether + lumefantrine - artesunate + amodiaquine - artesunate + mefloquine - artesunate + SP - dihydroartemisinin + piperaquine.",
        "section_name": "4.3.1 | ARTEMISININ-BASED COMBINATION THERAPY",
        "page": 37
    },
    {
        "question": "What is the recommended duration of treatment for an ACT regimen?",
        "answer_section_content": "ACT regimens should provide 3 days' treatment with an artemisinin derivative.",
        "section_name": "4.3.2 | DURATION OF TREATMENT",
        "page": 37
    },
    {
        "question": "What is the recommended treatment for uncomplicated P. falciparum malaria during the first trimester of pregnancy?",
        "answer_section_content": "Treat pregnant women with uncomplicated P. falciparum malaria during the first trimester with 7 days of quinine + clindamycin.",
        "section_name": "5.1.1 | FIRST TRIMESTER",
        "page": 52
    },
    {
        "question": "How should infants weighing less than 5kg with uncomplicated P. falciparum malaria be treated?",
        "answer_section_content": "Treat infants weighing < 5kg with uncomplicated P. falciparum malaria with an ACT at the same mg/kg bw target dose as for children weighing 5 kg.",
        "section_name": "5.2.2 | OPTIMAL ANTIMALARIAL DOSING IN INFANTS",
        "page": 56
    },
    {
        "question": "What is the recommended treatment for severe malaria for all patient groups, including pregnant women and infants?",
        "answer_section_content": "Treat adults and children with severe malaria (including infants, pregnant women in all trimesters and lactating women) with intravenous or intramuscular artesunate for at least 24 h and until they can tolerate oral medication. Once a patient has received at least 24 h of parenteral therapy and can tolerate oral therapy, complete treatment with 3 days of an ACT.",
        "section_name": "7 | Treatment of severe malaria",
        "page": 75
    },
    {
        "question": "What is the revised dose recommendation for parenteral artesunate in young children with severe malaria?",
        "answer_section_content": "Children weighing less than 20kg should receive a higher parenteral dose of artesunate (3 mg/kg/dose) than larger children and adults (2.4 mg/kg/dose) to ensure equivalent drug exposure.",
        "section_name": "7.4.1 | ARTESUNATE",
        "page": 80
    },
    {
        "question": "What is the recommended pre-referral treatment option for children under 6 years of age with suspected severe malaria?",
        "answer_section_content": "Where intramuscular injections of artesunate are not available, treat children < 6 years with a single rectal dose (10mg/kg bw) of artesunate, and refer immediately to an appropriate facility for further care. Do not use rectal artesunate in older children and adults.",
        "section_name": "7.5 | PRE-REFERRAL TREATMENT OPTIONS",
        "page": 84
    },
    {
        "question": "What is the recommended treatment for chloroquine-resistant P. vivax malaria?",
        "answer_section_content": "In areas with chloroquine-resistant infections, treat adults and children with uncomplicated P. vivax, P. ovale, P. malariae or P. knowlesi malaria (except pregnant women in their first trimester) with an ACT.",
        "section_name": "6.4 | TREATMENT OF BLOOD-STAGE INFECTION",
        "page": 66
    },
    {
        "question": "What is the recommended primaquine regimen to prevent relapse of P. vivax or P. ovale malaria?",
        "answer_section_content": "To prevent relapse, treat P. vivax or P. ovale malaria in children and adults (except pregnant women, infants aged < 6 months, women breastfeeding infants < 6 months, women breastfeeding older infants unless they are known not to be G6PD deficient and people with G6PD deficiency) with a 14-day course of primaquine in all transmission settings.",
        "section_name": "6.5 | TREATMENT OF THE LIVER STAGES (HYPNOZOITES) OF P. VIVAX AND P. OVALE",
        "page": 69
    },
    {
        "question": "What is the recommended preventive treatment for malaria in pregnancy in endemic areas of Africa?",
        "answer_section_content": "In malaria-endemic areas in Africa, provide SP-IPTp to all women in their first or second pregnancy as part of antenatal care. Dosing should start in the second trimester and doses should be given at least 1 month apart, with the objective of ensuring that at least three doses are received.",
        "section_name": "11.1 | INTERMITTENT PREVENTIVE TREATMENT OF MALARIA IN PREGNANCY WITH SULFADOXINE-PYRIMETHAMINE",
        "page": 104
    },
    {
        "question": "What is seasonal malaria chemoprevention and where is it recommended?",
        "answer_section_content": "In areas with highly seasonal malaria transmission in the sub-Sahel region of Africa, provide seasonal malaria chemoprevention (SMC) with monthly amodiaquine + SP for all children < 6 years during each transmission season.",
        "section_name": "11.3 | SEASONAL MALARIA CHEMOPREVENTION WITH AMODIAQUINE + SULFADOXINE-PYRIMETHAMINE",
        "page": 108
    },
    {
        "question": "What is the role of national drug and regulatory authorities regarding antimalarial drug quality?",
        "answer_section_content": "National drug and regulatory authorities should ensure that the antimalarial medicines provided in both the public and the private sectors are of acceptable quality, through regulation, inspection and law enforcement.",
        "section_name": "12 | QUALITY OF ANTIMALARIAL MEDICINES",
        "page": 111
    },
    {
        "question": "According to the Guidelines, what is the primary purpose of treating uncomplicated malaria?",
        "answer_section_content": "The clinical objectives of treating uncomplicated malaria are to cure the infection as rapidly as possible and to prevent progression to severe disease. 'Cure' is defined as elimination of all parasites from the body.",
        "section_name": "4.2 THERAPEUTIC OBJECTIVES",
        "page": 36
    },
    {
        "question": "What is the recommended single dose of primaquine to reduce the transmission of P. falciparum in low-transmission areas?",
        "answer_section_content": "In low-transmission areas, give a single dose of 0.25mg/kg bw primaquine with ACT to patients with P. falciparum malaria (except pregnant women, infants aged < 6 months and women breastfeeding infants aged < 6 months) to reduce transmission. G6PD testing is not required.",
        "section_name": "4.5 REDUCING THE TRANSMISSIBILITY OF TREATED P.FALCIPARUM INFECTIONS IN AREAS OF LOW-INTENSITY TRANSMISSION",
        "page": 45
    },
    {
        "question": "What is the definition of uncomplicated hyperparasitaemia and what are the associated risks?",
        "answer_section_content": "Uncomplicated hyperparasitaemia is present in patients who have ≥ 4% parasitaemia but no signs of severity. They are at increased risk for severe malaria and for treatment failure and are considered an important source of antimalarial drug resistance.",
        "section_name": "5.7 UNCOMPLICATED HYPERPARASITAEMIA",
        "page": 61
    }
]



In [17]:
pip install pandas


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [18]:
!pip install fuzzywuzzy
from fuzzywuzzy import fuzz
import pandas as pd

# Metric tracking
evaluation_results = []

for idx, sample in enumerate(test_data, start=1):
    query = sample["question"]
    expected_content = sample["answer_section_content"]
    expected_section = sample["section_name"]
    expected_page = sample["page"]

    # 1. Retrieve top-1 chunk from Chroma DB
    retrieved_docs = vector_store.similarity_search(query, k=1)

    if retrieved_docs:
        top_doc = retrieved_docs[0]
        retrieved_content = top_doc.page_content
        retrieved_page = top_doc.metadata.get("page")
        retrieved_section = top_doc.metadata.get("section")

        # 2. Compute Fuzzy Matching Score (token_set_ratio works best for partial text overlaps)
        content_fuzzy_score = fuzz.token_set_ratio(
            expected_content, retrieved_content
        )

        # 3. Check Page and Section equality
        page_is_same = retrieved_page == expected_page

        # Normalize strings for comparison (remove whitespace/lowercase)
        clean_expected_sec = str(expected_section).strip().lower()
        clean_retrieved_sec = str(retrieved_section).strip().lower()
        section_is_same = clean_expected_sec in clean_retrieved_sec or (
            clean_expected_sec == clean_retrieved_sec
        )

        evaluation_results.append(
            {
                "Sample": idx,
                "Fuzzy Score": content_fuzzy_score,
                "Page Match": page_is_same,
                "Section Match": section_is_same,
                "Retrieved Page": retrieved_page,
                "Expected Page": expected_page,
            }
        )

# Convert to DataFrame for easy viewing
df_results = pd.DataFrame(evaluation_results)

# Calculate Dataset Averages
avg_fuzzy_score = df_results["Fuzzy Score"].mean()
page_accuracy = df_results["Page Match"].mean() * 100
section_accuracy = df_results["Section Match"].mean() * 100

print(df_results.to_string(index=False))
print("\n" + "=" * 45)
print(f"Average Fuzzy Similarity Score: {avg_fuzzy_score:.2f}%")
print(f"Page Match Accuracy:          {page_accuracy:.2f}%")
print(f"Section Match Accuracy:       {section_accuracy:.2f}%")
print("=" * 45)


[notice] A new release of pip is available: 25.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip
c:\Users\dell\OneDrive\Desktop\c++ folder\Ai hacathon creativa\.venv-1\Lib\site-packages\fuzzywuzzy\fuzz.py:11: UserWarning: Using slow pure-python SequenceMatcher. Install python-Levenshtein to remove this warning
  warnings.warn('Using slow pure-python SequenceMatcher. Install python-Levenshtein to remove this warning')


 Sample  Fuzzy Score  Page Match  Section Match  Retrieved Page  Expected Page
      1           88       False          False              35             37
      2          100       False          False              35             37
      3           70       False          False              36             52
      4           93       False          False              51             56
      5           96       False          False              79             75
      6           98        True          False              80             80
      7           61       False          False              85             84
      8           90       False          False              63             66
      9           97        True          False              69             69
     10          100       False          False             103            104
     11          100       False          False             103            108
     12          100       False          False     

In [19]:
import pandas as pd
from fuzzywuzzy import fuzz

# List container to hold all evaluated samples
results_container = []

for idx, sample in enumerate(test_data):
    query = sample["question"]
    expected_content = sample["answer_section_content"]
    expected_section = sample["section_name"]
    expected_page = sample["page"]

    # Retrieve top-1 chunk from Chroma DB
    retrieved_docs = vector_store.similarity_search(query, k=1)

    if retrieved_docs:
        top_doc = retrieved_docs[0]
        retrieved_content = top_doc.page_content
        retrieved_page = top_doc.metadata.get("page")
        retrieved_section = top_doc.metadata.get("section")

        # Fuzzy matching score
        content_fuzzy_score = fuzz.token_set_ratio(
            expected_content, retrieved_content
        )

        # Equality checks
        page_is_same = retrieved_page == expected_page

        clean_expected_sec = str(expected_section).strip().lower()
        clean_retrieved_sec = str(retrieved_section).strip().lower()
        section_is_same = (
            clean_expected_sec in clean_retrieved_sec
            or clean_expected_sec == clean_retrieved_sec
        )

        # Append detailed record to container
        results_container.append(
            {
                "sample_id": idx,
                "question": query,
                "fuzzy_score": content_fuzzy_score,
                "page_match": page_is_same,
                "section_match": section_is_same,
                "expected_content": expected_content,
                "retrieved_content": retrieved_content,
                "expected_metadata": {
                    "page": expected_page,
                    "section": expected_section,
                },
                "retrieved_metadata": {
                    "page": retrieved_page,
                    "section": retrieved_section,
                },
            }
        )

# Sort container by Fuzzy Score (highest to lowest)
results_container = sorted(
    results_container, key=lambda x: x["fuzzy_score"], reverse=True
)

# Print Summary Averages
df_summary = pd.DataFrame(results_container)
print(f"Total Samples Evaluated: {len(results_container)}")
print(f"Average Fuzzy Score:    {df_summary['fuzzy_score'].mean():.2f}%")
print(f"Page Match Accuracy:    {df_summary['page_match'].mean() * 100:.2f}%")
print(f"Section Match Accuracy: {df_summary['section_match'].mean() * 100:.2f}%")

Total Samples Evaluated: 15
Average Fuzzy Score:    87.13%
Page Match Accuracy:    26.67%
Section Match Accuracy: 0.00%


In [20]:
results_container[0]

{'sample_id': 1,
 'question': 'What is the recommended duration of treatment for an ACT regimen?',
 'fuzzy_score': 100,
 'page_match': False,
 'section_match': False,
 'expected_content': "ACT regimens should provide 3 days' treatment with an artemisinin derivative.",
 'retrieved_content': 'ACT regimens should provide 3 days’ treatment with an artemisinin derivative. _Strong recommendation, high-quality evidence_',
 'expected_metadata': {'page': 37, 'section': '4.3.2 | DURATION OF TREATMENT'},
 'retrieved_metadata': {'page': 35, 'section': '_Duration of ACT treatment_'}}

In [21]:
import chromadb
from fastembed import TextEmbedding

client = chromadb.PersistentClient(path="src/vectorstore/chroma")  # Ensure this path is correct and points to your ChromaDB directory
collection = client.get_or_create_collection("who_malaria_guidelines")
embedder = TextEmbedding(model_name="BAAI/bge-small-en-v1.5")

def ingest(chunks: list[dict]):
    texts = [c["text"] for c in chunks]
    embeddings = list(embedder.embed(texts))

    collection.add(
        ids=[f"chunk_{i}" for i in range(len(chunks))],
        embeddings=[e.tolist() for e in embeddings],
        documents=texts,
        metadatas=[
            {
                "page": c["page"],
                "section": c["section"] or "unknown",
                "has_table": "|---|" in c["text"],  # cheap table flag
            }
            for c in chunks
        ],
    )


ingest(chunks)

In [22]:
collection.peek()

{'ids': ['chunk_0',
  'chunk_1',
  'chunk_2',
  'chunk_3',
  'chunk_4',
  'chunk_5',
  'chunk_6',
  'chunk_7',
  'chunk_8',
  'chunk_9'],
 'embeddings': array([[-0.01357071,  0.0130882 ,  0.02479587, ..., -0.01771899,
          0.06851702,  0.01935418],
        [-0.01743889,  0.01249254,  0.01623897, ..., -0.04405054,
          0.07199543, -0.01977208],
        [-0.03047764,  0.01214533, -0.0237661 , ..., -0.0511637 ,
          0.0248959 , -0.03440503],
        ...,
        [-0.01327157,  0.05494801,  0.02304572, ...,  0.00346085,
          0.0919701 ,  0.01390988],
        [ 0.0024969 ,  0.04733192,  0.05137987, ..., -0.01408846,
          0.07593215,  0.01710458],
        [-0.01501577,  0.04663565,  0.01442849, ..., -0.0564326 ,
          0.04241788,  0.0071742 ]], shape=(10, 384)),
 'documents': ['**Artemisinin-based combination therapy (ACT).** A combination of an artemisinin derivative with a longer-acting antimalarial that has a different mode of action. **Asexual cycle.** The li

In [23]:
# --- Central Configuration ---
import src.config as config
from src.config import CHUNK_OVERLAP, CHUNK_SIZE, DOCUMENTS_DIR, EMBEDDING_MODEL, LLM_MODEL

# --- Core Utility Wrappers ---
from src.embedder import Embedder, embedder

# --- Data Ingestion Pipeline ---
from src.ingestion.chunker import chunk_documents  # adjust function names if different
from src.ingestion.ingest import run_ingestion
from src.ingestion.parser import load_documents

# --- Vector Store & Storage ---
from src.vectorstore.vector_store import VectorStore, vector_store

# --- Retrieval & Evaluation ---
from src.retrieval.evaluator import evaluate_with_ranx
from src.retrieval.retriever import format_context, retrieve, retrieve_cosine_similarity

# --- LLM Generation ---
from src.generation.generator import generate_answer

# --- FastAPI App & Schemas ---
from src.api.routes import router
from src.api.schemas import ChatRequest, ChatResponse
from src.main import app as fastapi_app
from src.retrieval.expand_retrieval import expand_context

ModuleNotFoundError: No module named 'ranx'